# IMPORT THƯ VIỆN

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from __future__ import print_function, division

from keras.layers import Input, Dropout, Concatenate, ConvLSTM2D, BatchNormalization, LeakyReLU, UpSampling2D, Conv2D, MaxPooling2D
from keras.layers import Dense, Flatten, Reshape
from keras.layers import Lambda
from keras.models import Model, load_model
from keras.optimizers import Adam
import tensorflow as tf
from keras.models import load_model
import numpy as np
from tqdm import tqdm
tf.config.optimizer.set_experimental_options({"remapping": False})

import os
import cv2
import math
import datetime
import numpy as np
import matplotlib.pyplot as plt

# TIỀN XỬ LÝ DỮ LIỆU

In [ ]:
# ------------------------------
# Chuyển ảnh về ma trận ảnh Gray và resize
# ------------------------------
def matrix_images(image_folder, image_files, image_size=(224, 224)):
    matrix_img = []
    for img_file in image_files:
        img_path = os.path.join(image_folder, img_file)
        try:
            image = cv2.imread(img_path)
            if image is None:
                raise ValueError(f"Không thể đọc ảnh: {img_file}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            # Resize ảnh
            image = cv2.resize(image, image_size)
            matrix_img.append(image)
        except Exception as e:
            print(f"Lỗi xử lý ảnh {img_file}: {e}")
    return matrix_img

In [ ]:
# ------------------------------
# Chuẩn hóa dữ liệu
# ------------------------------
def scaling_img(data_img):
    # Chuyển danh sách các mảng numpy thành một mảng numpy đa chiều
    data_array = np.array(data_img, dtype=np.float32)

    # Chuẩn hóa dữ liệu
    scaled_img = data_array / 255.0
    return scaled_img

In [ ]:
# ------------------------------
# Giảm nhiễu
# ------------------------------
def calculate_sigma(kernel_size):
    sigma = (kernel_size - 1) / 6.0
    return sigma

def reduce_noise(image, kernel_size=3):
    sigma = calculate_sigma(kernel_size)
    smoothed_image = cv2.GaussianBlur(image, (kernel_size, kernel_size), sigmaX=sigma)
    return smoothed_image

In [ ]:
# ------------------------------
# Tạo nhãn chuỗi dữ liệu các ảnh liên tiếp từ dữ liệu hình ảnh
# ------------------------------
def create_image_sequences(data, time_steps):
    num_samples, height, width, channels = data.shape
    num_frames = num_samples - time_steps
    input_sequences = np.zeros((num_frames, time_steps, height, width, channels), dtype=np.float32)
    labels = np.zeros((num_frames, height, width, channels), dtype=np.float32)

    for i in range(num_frames):
        input_sequences[i] = data[i:i+time_steps]
        labels[i] = data[i+time_steps]

    return input_sequences, labels

In [ ]:
def create_seq2img(data, time_steps_input=3, time_steps_output=6):
    N, C, H, W = data.shape
    M = N - time_steps_input - time_steps_output + 1
    if M <= 0:
        raise ValueError("Không đủ khung hình để tạo chuỗi")
    X = np.stack([data[i : i + time_steps_input] for i in range(M)])
    Y = np.stack([data[i + time_steps_input : i + time_steps_input + time_steps_output] for i in range(M)])
    return X.astype(np.float32), Y.astype(np.float32)

In [ ]:
# ------------------------------
# Chia dữ liệu
# ------------------------------
def Split_Data(X_data, y_data):
    # Chia dữ liệu thành các tập train, val, test
    size = int(len(X_data) * 0.8)
    size_val = int((len(X_data) - size) / 2)

    X_train = X_data[:size]
    X_val = X_data[size:size + size_val]
    X_test = X_data[size + size_val:]

    y_train = y_data[:size]
    y_val = y_data[size:size + size_val]
    y_test = y_data[size + size_val:]

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
# ------------------------------
# Chuyển định dạng dữ liệu (B, T, H, W, C) => (B, H, W, T)
# ------------------------------
def reshape_seq_img(X_data):
    # X_data hiện tại có dạng (B, T, H, W, C)
    X_data_new = np.transpose(X_data, (0, 2, 3, 1, 4))  # Hoán đổi trục
    X_data_new = X_data_new.reshape(X_data_new.shape[0], X_data_new.shape[1], X_data_new.shape[2], -1)

    return X_data_new

def reshape_BHWT_BTHWC(shape_tuple):
    if len(shape_tuple) == 4:
        B, H, W, T = shape_tuple
        new_shape = (B, T, H, W, 1)
    elif len(shape_tuple) == 3:
        H, W, T = shape_tuple
        new_shape = (T, H, W, 1)
    else:
        raise ValueError("Input shape phải có phần tử (B, H, W, T) hoặc (H, W, T), nhận được: {}".format(shape_tuple))
    return new_shape

# XÂY DỰNG MÔ HÌNH Radar2Radar_GAN

In [ ]:
from keras.layers import *

def generator_model(image_shape=(224, 224, 4)):
    inputs = Input(shape=image_shape)

    conv1s = Conv2D(128, 3, padding="same", kernel_initializer="he_normal")(inputs)
    bn1s = BatchNormalization()(conv1s)
    act1s = Activation("relu")(bn1s)
    pool1 = MaxPooling2D(pool_size=(2, 2))(act1s)
    drop1 = Dropout(0.5)(pool1)

    conv2f = Conv2D(256, 3, padding="same", kernel_initializer="he_normal")(drop1)
    bn2f = BatchNormalization()(conv2f)
    act2f = Activation("relu")(bn2f)
    conv2s = Conv2D(512, 3, padding="same", kernel_initializer="he_normal")(act2f)
    bn2s = BatchNormalization()(conv2s)
    act2s = Activation("relu")(bn2s)
    pool2 = MaxPooling2D(pool_size=(2, 2))(act2s)
    drop2 = Dropout(0.5)(pool2)

    conv3f = Conv2D(1024, 3, padding="same", kernel_initializer="he_normal")(drop2)
    bn3f = BatchNormalization()(conv3f)
    act3f = Activation("relu")(bn3f)
    drop3 = Dropout(0.5)(act3f)

    up4 = concatenate([UpSampling2D(size=(2, 2))(drop3), act2s], axis=3)
    conv4f = Conv2D(512, 3, padding="same", kernel_initializer="he_normal")(up4)
    bn4f = BatchNormalization()(conv4f)
    act4f = Activation("relu")(bn4f)
    drop4f = Dropout(0.5)(act4f)
    conv4 = Conv2D(256, 3, padding="same", activation="relu", kernel_initializer="he_normal")(drop4f)
    bn4 = BatchNormalization()(conv4)
    act4 = Activation("relu")(bn4)

    up5 = concatenate([UpSampling2D(size=(2, 2))(act4), act1s], axis=3)
    conv5f = Conv2D(128, 3, padding="same", kernel_initializer="he_normal")(up5)
    bn5f = BatchNormalization()(conv5f)
    act5f = Activation("relu")(bn5f)
    drop5f = Dropout(0.5)(act5f)
    conv5s = Conv2D(64, 3, padding="same", kernel_initializer="he_normal")(drop5f)
    bn5s = BatchNormalization()(conv5s)
    act5s = Activation("relu")(bn5s)
    drop5s = Dropout(0.5)(act5s)
    conv5 = Conv2D(2, 3, padding="same", kernel_initializer="he_normal")(drop5s)
    bn5 = BatchNormalization()(conv5)
    act5 = Activation("relu")(bn5)

    outputs = Conv2D(1, 1, activation="sigmoid")(act5)

    model = Model(inputs=inputs, outputs=outputs)
    return model


In [ ]:
def discriminate_model(lr, input_shape=(224, 224, 4), output_shape=(224, 224, 1)):
    in_inputs = Input(shape=input_shape, name="input")
    tar_inputs = Input(shape=output_shape, name="target")

    concat = concatenate([in_inputs, tar_inputs], axis=3)

    conv1 = Conv2D(64, 4, strides=(2, 2), padding="same", kernel_initializer="he_normal")(concat)
    bn1 = BatchNormalization()(conv1)
    act1 = LeakyReLU(alpha=0.2)(bn1)

    conv2 = Conv2D(128, 4, strides=(2, 2), padding="same", kernel_initializer="he_normal")(act1)
    bn2 = BatchNormalization()(conv2)
    act2 = LeakyReLU(alpha=0.2)(bn2)

    conv4 = Conv2D(256, 4, strides=(1, 1), padding="same", kernel_initializer="he_normal")(act2)
    bn4 = BatchNormalization()(conv4)
    act4 = LeakyReLU(alpha=0.2)(bn4)

    conv = Conv2D(1, 4, padding="same", kernel_initializer="he_normal")(act4)
    outputs = Activation("sigmoid")(conv)

    model = Model([in_inputs, tar_inputs], outputs)

    opt = Adam(learning_rate=lr, beta_1=0.5)
    model.compile(loss="binary_crossentropy", optimizer=opt, loss_weights=[0.5])
    return model


In [ ]:
def gan_model(lr, generator, discriminator, input_shape=(224, 224, 4)):
    discriminator.trainable = False

    inputs = Input(shape=input_shape)
    gen_out = generator(inputs)
    dis_out = discriminator([inputs, gen_out])

    model = Model(inputs, [dis_out, gen_out])

    opt = Adam(learning_rate=lr)
    model.compile(loss=["binary_crossentropy", "mae"], optimizer=opt, loss_weights=[1, 100])
    return model

In [ ]:
def train_gan_model(generator, discriminator, gan, train_data, val_data, epochs=100, batch_size=16, path_weight="rad_cgan_best_generator.keras"):
    X_train, Y_train = train_data
    X_val, Y_val = val_data

    half_batch = batch_size // 2
    best_val_loss = float('inf')

    size_img_ = int(X_train.shape[1]/4)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        for i in tqdm(range(len(X_train) // batch_size)):
            # === Huấn luyện Discriminator ===
            idx = np.random.randint(0, X_train.shape[0], half_batch)
            imgs_in = X_train[idx]
            real_imgs = Y_train[idx]

            fake_imgs = generator.predict(imgs_in)

            valid = np.ones((half_batch, size_img_, size_img_, 1))
            fake = np.zeros((half_batch, size_img_, size_img_, 1))

            d_loss_real = discriminator.train_on_batch([imgs_in, real_imgs], valid)
            d_loss_fake = discriminator.train_on_batch([imgs_in, fake_imgs], fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # === Huấn luyện Generator ===
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            imgs_in = X_train[idx]
            real_imgs = Y_train[idx]

            valid = np.ones((batch_size, size_img_, size_img_, 1))
            g_loss = gan.train_on_batch(imgs_in, [valid, real_imgs])

        # === Đánh giá trên validation set ===
        val_preds = generator.predict(X_val)
        val_loss = np.mean(np.abs(val_preds - Y_val))  # MAE thủ công

        print(f"[Epoch {epoch+1}] [D loss: {d_loss:.4f}] [G loss: {g_loss}] [val MAE: {val_loss:.6f}]")

        # === Lưu generator tốt nhất ===
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            generator.save(path_weight)
            print(f"✅ Saved best generator at epoch {epoch+1} with val_loss: {val_loss:.6f}")

# ĐÁNH GIÁ MÔ HÌNH VÀ TRỰC QUAN KẾT QUẢ

### Đánh giá kết quả

In [ ]:
import numpy as np
import torch
from skimage.metrics import structural_similarity as ssim

In [ ]:
def evaluate_image_metrics(y_test, predictions, value_range=(0.0, 1.0)):
    mse_list = []
    mae_list = []
    rmse_list = []
    rmae_list = []
    ssim_list = []

    num_samples = y_test.shape[0]
    min_val, max_val = value_range
    data_range = max_val - min_val if (max_val - min_val) > 0 else 1.0

    for i in range(num_samples):
        # Lấy ảnh ground truth và dự đoán
        y_true_img = y_test[i].astype(np.float32)
        y_pred_img = predictions[i].astype(np.float32)

        # Nếu shape là (H, W, 1), ta squeeze về (H, W)
        if y_true_img.ndim == 3 and y_true_img.shape[2] == 1:
            y_true_img = np.squeeze(y_true_img, axis=2)
        if y_pred_img.ndim == 3 and y_pred_img.shape[2] == 1:
            y_pred_img = np.squeeze(y_pred_img, axis=2)

        # Bây giờ y_true_img và y_pred_img nên có shape (H, W) cho grayscale.
        # Nếu shape khác (ví dụ H, W, C với C>1) thì có thể mở rộng sau này.

        # Flatten để tính MSE/MAE/...
        y_true_flat = y_true_img.flatten()
        y_pred_flat = y_pred_img.flatten()

        # Tính MSE, MAE, RMSE
        mse = np.mean((y_true_flat - y_pred_flat) ** 2)
        mae = np.mean(np.abs(y_true_flat - y_pred_flat))
        rmse = np.sqrt(mse)

        # Tính RMAE = MAE / mean(|y_true|), tránh chia cho 0
        mean_abs_y = np.mean(np.abs(y_true_flat))
        rmae = mae / mean_abs_y if mean_abs_y != 0 else np.nan

        # Tính SSIM cho grayscale
        try:
            # y_true_img và y_pred_img giờ là 2D (H, W)
            ssim_val = ssim(
                y_true_img,
                y_pred_img,
                data_range=data_range
            )
        except ValueError:
            # Nếu SSIM lỗi (ví dụ data_range không phù hợp),
            # tính lại data_range tự động từ dữ liệu thực tế
            dmin = min(float(y_true_img.min()), float(y_pred_img.min()))
            dmax = max(float(y_true_img.max()), float(y_pred_img.max()))
            auto_range = dmax - dmin if (dmax - dmin) > 0 else 1.0
            ssim_val = ssim(
                y_true_img,
                y_pred_img,
                data_range=auto_range
            )

        mse_list.append(mse)
        mae_list.append(mae)
        rmse_list.append(rmse)
        rmae_list.append(rmae)
        ssim_list.append(ssim_val)

    metrics_avg = {
        'MSE': np.mean(mse_list),
        'MAE': np.mean(mae_list),
        'RMSE': np.mean(rmse_list),
        'RMAE': np.nanmean(rmae_list),
        'SSIM': np.mean(ssim_list)
    }

    return metrics_avg

### Trực quan hóa ảnh dự báo

In [ ]:
def display_gt_pred_rows(y_test_uint8, predictions_img, channels=1):
    num_samples = y_test_uint8.shape[0]
    # Tạo figure với 2 hàng và num_samples cột
    fig, axes = plt.subplots(2, num_samples, figsize=(num_samples * 4, 2 * 4))

    # Nếu chỉ có 1 mẫu, đảm bảo axes có kích thước 2 x 1
    if num_samples == 1:
        axes = np.reshape(axes, (2, 1))

    # Hàng trên: hiển thị ảnh ground truth
    for i in range(num_samples):
        ax = axes[0, i]
        if channels == 1:
            ax.imshow(y_test_uint8[i].squeeze())
        else:
            ax.imshow(y_test_uint8[i])
        ax.set_title(f"GT {i}")
        ax.axis('off')

    # Hàng dưới: hiển thị ảnh dự đoán
    for i in range(num_samples):
        ax = axes[1, i]
        if channels == 1:
            ax.imshow(predictions_img[i].squeeze())
        else:
            ax.imshow(predictions_img[i])
        ax.set_title(f"Pred {i}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# THỰC HIỆN CHẠY MÔ HÌNH

### Đọc dữ liệu

In [ ]:
# Đọc từng file ảnh và sắp xếp theo thứ tự tên tệp
image_folder = r'/content/drive/MyDrive/BacSon/Data_Duc'
image_files = sorted([f for f in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, f))],
                     key=lambda x: int(''.join(filter(str.isdigit, x))) if any(c.isdigit() for c in x) else x)

In [ ]:
# params
timesteps = 4
im_width = im_height = 800
channels = 1

In [ ]:
# Resize và chuyển đổi kênh màu
data_img = matrix_images(image_folder, image_files[-64:], image_size=(im_width, im_height))

if len(data_img) == 0:
    raise ValueError("Không có ảnh nào được xử lý, vui lòng kiểm tra lại thư mục!")

# Thực hiện chuẩn hóa và khử nhiễu dữ liệu
data_img_processing = []
data_img_scaled = scaling_img(data_img)
for img in data_img_scaled:
    data_noise_img = reduce_noise(img, kernel_size=3)
    data_img_processing.append(data_noise_img)

# Chuyển thành numpy array và thêm trục kênh
data_image = np.expand_dims(np.array(data_img_processing), axis=-1)

# Tạo chuỗi dữ liệu ảnh liên tiếp
datas, labels = create_image_sequences(data_image, timesteps)

seq_data, seq_label = create_seq2img(data_image, time_steps_input=timesteps, time_steps_output=6)

# Chia dữ liệu
X_train, y_train, X_val, y_val, X_test, y_test = Split_Data(datas, labels)
X_train = reshape_seq_img(X_train)
X_val = reshape_seq_img(X_val)
X_test = reshape_seq_img(X_test)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print()
_, _, _, _, X_test_seq, y_test_seq = Split_Data(seq_data, seq_label)
X_test_seq = reshape_seq_img(X_test_seq)
y_test_seq = reshape_seq_img(y_test_seq)
print("X_test_seq shape:", X_test_seq.shape)
print("y_test_seq shape:", y_test_seq.shape)

### Thực hiện huấn luyện mô hình

In [ ]:
# Tạo mô hình generator
build_generator = generator_model(image_shape=(im_width, im_height, timesteps))
# Tạo mô hình discriminator
build_discriminator = discriminate_model(lr=1e-4,
                                         input_shape=(im_width, im_height, timesteps),
                                         output_shape=(im_width, im_height, channels))
# Tạo mô hình GAN
build_gan = gan_model(1e-4, build_generator, build_discriminator, input_shape=(im_width, im_height, timesteps))

In [ ]:
train_gan_model(build_generator, build_discriminator, build_gan,
                train_data=(X_train, y_train),
                val_data=(X_val, y_val),
                epochs=200,
                batch_size=4,
                path_weight='/content/drive/MyDrive/BacSon/MoNet-2025/Rad-cGAN_DataDuc.keras')

In [ ]:
# Đường dẫn đến file mô hình đã lưu
saved_model_path = '/content/drive/MyDrive/BacSon/MoNet-2025/Rad-cGAN_DataDuc.keras'

# Load mô hình đã lưu
generator = load_model(saved_model_path)

# Dự đoán bằng mô hình generator
predictions_img = generator.predict(X_test, batch_size=1)

In [ ]:
# Tính toán các chỉ số đánh giá
metrics_avg = evaluate_image_metrics(y_test, predictions_img)

print("Các chỉ số trung bình trên tập ảnh:")
for metric, value in metrics_avg.items():
    print(f"{metric}: {value:.6f}")

### Trực quan ảnh thực tế và ảnh dự đoán

In [ ]:
# Trực quan y_test_uint8 và predictions_img
display_gt_pred_rows(y_test, predictions_img, channels=1)

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from skimage.metrics import structural_similarity as ssim

# --- 1. Load model ---
saved_model_path = '/content/drive/MyDrive/BacSon/MoNet-2025/Rad-cGAN_DataDuc.keras'

generator = load_model(saved_model_path)

# --- 3. Đổi Y_test_seq về (n, 6, H, W, 1) ---
Y_test_seq = np.moveaxis(y_test_seq, -1, 1)[..., np.newaxis]

# --- 4. Thiết lập ---
n, H, W, _ = X_test.shape
horizon = 6

# --- 5. Dự đoán multi-step ---
preds_seq = np.zeros((n, horizon, H, W, 1), dtype=np.float32)
for i in range(n):
    window = X_test[i].copy()
    for t in range(horizon):
        inp = window[None, ...]
        next_frame = generator.predict(inp, batch_size=1)
        preds_seq[i, t] = next_frame[0]
        window = np.concatenate([window[..., 1:], next_frame[0]], axis=-1)

# --- 6. Hàm đánh giá ---
import numpy as np
from skimage.metrics import structural_similarity as ssim
import torch
from scipy.linalg import sqrtm
from torchvision.models import inception_v3
from torchvision.transforms import Resize, Normalize, Compose

class FIDCalculator:
    def __init__(self, device='cuda'):
        self.device = device
        self.model = inception_v3(pretrained=True, transform_input=False)
        self.model.fc = torch.nn.Identity()
        self.model.to(device).eval()
        self.transform = Compose([
            Resize((299, 299)),
            Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def get_activations(self, imgs):
        with torch.no_grad():
            act = self.model(imgs.to(self.device))
        return act.cpu().numpy()

    @staticmethod
    def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
        diff = mu1 - mu2
        covmean = sqrtm(sigma1.dot(sigma2))
        if not np.isfinite(covmean).all():
            offset = np.eye(sigma1.shape[0]) * eps
            covmean = sqrtm((sigma1 + offset).dot(sigma2 + offset))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

    def compute_fid(self, preds, trues):
        # preds, trues shape (N, H, W)
        N, H, W = preds.shape
        p_imgs = torch.tensor(preds, dtype=torch.float32).unsqueeze(1).repeat(1,3,1,1)
        t_imgs = torch.tensor(trues, dtype=torch.float32).unsqueeze(1).repeat(1,3,1,1)
        p_in = self.transform(p_imgs)
        t_in = self.transform(t_imgs)
        f_p = self.get_activations(p_in)
        f_t = self.get_activations(t_in)
        mu_p, sigma_p = np.mean(f_p, axis=0), np.cov(f_p, rowvar=False)
        mu_t, sigma_t = np.mean(f_t, axis=0), np.cov(f_t, rowvar=False)
        return self.calculate_frechet_distance(mu_p, sigma_p, mu_t, sigma_t)


def evaluate_image_metrics(y_test, predictions, value_range=(0.0, 1.0), threshold_norm=30/255, device='cuda'):
    # Chuyển và squeeze channel nếu cần
    y = np.array(y_test)
    p = np.array(predictions)
    if y.ndim == 4 and y.shape[-1] == 1:
        y = y[..., 0]
    if p.ndim == 4 and p.shape[-1] == 1:
        p = p[..., 0]

    N = y.shape[0]
    min_v, max_v = value_range
    data_range = max_v - min_v if (max_v - min_v) > 0 else 1.0

    mse_list, mae_list, rmse_list, rmae_list, ssim_list, csi_list = [], [], [], [], [], []
    for yt, yp in zip(y, p):
        yt, yp = yt.astype(np.float32), yp.astype(np.float32)
        yt_flat, yp_flat = yt.ravel(), yp.ravel()

        # MSE, MAE, RMSE
        mse = np.mean((yt_flat - yp_flat)**2)
        mae = np.mean(np.abs(yt_flat - yp_flat))
        rmse = np.sqrt(mse)
        # RMAE
        mean_abs = np.mean(np.abs(yt_flat))
        rmae = mae/mean_abs if mean_abs != 0 else np.nan

        # SSIM
        try:
            s_val = ssim(yt, yp, data_range=data_range)
        except ValueError:
            auto_range = max(yt.max(), yp.max()) - min(yt.min(), yp.min())
            s_val = ssim(yt, yp, data_range=auto_range if auto_range>0 else 1.0)

        # CSI@30dBZ
        p_bin = yp >= threshold_norm
        g_bin = yt >= threshold_norm
        tp = np.logical_and(p_bin, g_bin).sum()
        fn = np.logical_and(~p_bin, g_bin).sum()
        fp = np.logical_and(p_bin, ~g_bin).sum()
        csi = tp / (tp + fn + fp + 1e-8)

        mse_list.append(mse)
        mae_list.append(mae)
        rmse_list.append(rmse)
        rmae_list.append(rmae)
        ssim_list.append(s_val)
        csi_list.append(csi)

    # Trung bình metrics
    metrics_avg = {
        'MSE': np.mean(mse_list),
        'MAE': np.mean(mae_list),
        'RMSE': np.mean(rmse_list),
        'RMAE': np.nanmean(rmae_list),
        'SSIM': np.mean(ssim_list),
        'CSI@30dBZ': np.mean(csi_list)
    }
    # Tính FID
    fid_calc = FIDCalculator(device)
    fid_val = fid_calc.compute_fid(p, y)
    metrics_avg['FID'] = fid_val
    return metrics_avg


# --- 7. Tính và in metrics với 5 chữ số ---
all_metrics = []
for t in range(horizon):
    m = evaluate_image_metrics(Y_test_seq[:, t], preds_seq[:, t])
    all_metrics.append(m)
    print(f"Frame t+{t+1}: "
          f"MSE={m['MSE']:.5f}, "
          f"MAE={m['MAE']:.5f}, "
          f"RMSE={m['RMSE']:.5f}, "
          f"RMAE={m['RMAE']:.5f}, "
          f"SSIM={m['SSIM']:.5f}",
          f"CSI={m['CSI@30dBZ']:.5f}",
          f"FID={m['FID']:.5f}")

# Trung bình qua tất cả frames
avg = {k: float(np.mean([m[k] for m in all_metrics])) for k in all_metrics[0]}
print("\nAverage over t+1…t+6: "
      f"MSE={avg['MSE']:.5f}, "
      f"MAE={avg['MAE']:.5f}, "
      f"RMSE={avg['RMSE']:.5f}, "
      f"RMAE={avg['RMAE']:.5f}, "
      f"SSIM={avg['SSIM']:.5f}, ",
      f"CSI={avg['CSI@30dBZ']:.5f}, ",
        f"FID={avg['FID']:.5f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Chọn index của mẫu muốn hiển thị
sample_idx = 0  # có thể đổi sang mẫu khác
frames_to_plot = horizon  # số frame muốn hiển thị

fig, axes = plt.subplots(2, frames_to_plot, figsize=(frames_to_plot * 3, 6))

for t in range(frames_to_plot):
    # Ground truth
    gt_frame = Y_test_seq[sample_idx, t, ..., 0]
    axes[0, t].imshow(gt_frame, cmap='jet', vmin=0, vmax=1)
    axes[0, t].set_title(f"GT t+{t+1}")
    axes[0, t].axis('off')

    # Prediction
    pred_frame = preds_seq[sample_idx, t, ..., 0]
    axes[1, t].imshow(pred_frame, cmap='jet', vmin=0, vmax=1)
    axes[1, t].set_title(f"Pred t+{t+1}")
    axes[1, t].axis('off')

plt.tight_layout()
plt.show()
